# AI Career Coach: end-to-end data and model pipeline

This notebook runs the repository pipeline from Google Colab. Copy the project folder and `Datasets` folder into Google Drive first, keeping the folder names shown below. The notebook prepares separate O*NET occupation profiles and LinkedIn market-demand tables. Classifier and RAG retriever training are optional and remain separate.

Job postings describe the labor market; they are not labeled examples of which career suits a student. The current LinkedIn export is U.S.-focused, so do not interpret its counts as Kenyan demand. No Gemini API key is needed for this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Change this if you placed the folders elsewhere in Google Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/AICareerCoach')
DATASET_DIR = PROJECT_DIR / 'Datasets'
RUNNER = PROJECT_DIR / 'ml' / 'src' / 'run_career_pipeline.py'
assert RUNNER.is_file(), f'Missing project code: {RUNNER}'
assert DATASET_DIR.is_dir(), f'Missing dataset folder: {DATASET_DIR}'
print('Project:', PROJECT_DIR)
for path in sorted(DATASET_DIR.glob('*')):
    if path.is_file(): print(f'{path.name}: {path.stat().st_size:,} bytes')

## Choose pipeline stages

Data preparation runs every time. Keep model training off while reviewing the data or when adding new sources. The classifier uses the labeled file already in the project; it does not learn student fit from job postings. Promotion replaces the backend model, so leave it off until you have reviewed the held-out metrics. Retriever fine-tuning is a separate task and requires its own data and dependencies.

In [ ]:
RUN_CAREER_CLASSIFIER = False
PROMOTE_MODEL = False
RUN_RAG_RETRIEVER = False

if PROMOTE_MODEL and not RUN_CAREER_CLASSIFIER:
    raise ValueError('PROMOTE_MODEL requires RUN_CAREER_CLASSIFIER=True')

In [ ]:
# The classifier's dependencies are installed only when training is enabled.
if RUN_CAREER_CLASSIFIER:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'joblib'], check=True)

if RUN_RAG_RETRIEVER:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'torch', 'pandas', 'numpy', 'matplotlib'], check=True)

command = [sys.executable, str(RUNNER), '--input-dir', str(DATASET_DIR)]
if RUN_CAREER_CLASSIFIER: command.append('--train-classifier')
if PROMOTE_MODEL: command.append('--promote-model')
if RUN_RAG_RETRIEVER: command.append('--train-retriever')
subprocess.run(command, cwd=PROJECT_DIR, check=True)

## Review outputs

Derived data is written to `ml/data/processed/derived/`. Every classifier training run saves a versioned model and metrics under `ml/models/career-model/` and `ml/metrics/career-model/`. Compare validation and held-out test metrics between runs; do not treat model scores as student success probabilities. Keep original downloaded files unchanged and retain their source/licensing notes.